We will try to do simple Logistics Regression using the automated tagging system we have. 

In [36]:
import sys
import os

sys.path.append(os.path.abspath('../'))

from tagging_system import automate_tagging as auto_tag

In [37]:
import numpy as np
import pandas as pd

In [38]:
books = pd.read_csv('../data/data_with_author_and_awards.csv',
dtype = {
    'isbn' : 'str', # do this explicitly to avoide getting a warning by the interpreter
    'author_birthyear' : 'Int64', # we have to explicitly do this to avoid pandas implicitly casting as float
    'title_id' : 'Int64'
},
)
books = books.dropna() # we drop the books without descriptions because these are very, very unlikely to be winners any

In [39]:
tags = auto_tag.load_tags()
transformer = auto_tag.load_transformer()
tags_encoded = auto_tag.encode_tags()
tags_map = auto_tag.tag_dictionary()

In [40]:
books = auto_tag.encode_books(books, transformer) # takes a while

In [41]:
books = auto_tag.tag_books(books, transformer, tags_encoded)

In [55]:
books['target'] = books.hugo | books.locus

In [43]:
books['num_prev_awards'] = books.Hugo_Awards_Previously + books.Locus_Awards_Previously

In [44]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score

In [45]:
log_reg = LogisticRegression()
log_reg_balanced = LogisticRegression(class_weight='balanced')

In [56]:
# manually train test split our data

books_train = books[books.release_year <= 2014]
books_test = books[books.release_year <= 2014]

books_tt = books_train[books_train.release_year <= 2005]
books_val = books_train[books_train.release_year > 2005]

In [47]:
books_tt.head()

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,...,Hugo_Awards_Previously,Locus_Awards_Previously,Hugo_Nominee_Before,Locus_Nominee_Before,author_birthplace_country,author_birthplace_continent,encoded_synopsis,auto_tags,feature,num_prev_awards
0,9372,The Long Loud Silence,Wilson Tucker,1954,1954-00-00,Dell,1914,"Deer Creek, Illinois, USA",0899683754,Publisher's description: Tomorrow's war -- the...,...,0,0,False,False,USA,Central/North America,"[-0.041223828, 0.12155522, -0.006339091, 0.019...","[0.08351084, 0.0629492, 0.09347064, 0.14734189...",False,0
4,1908,The Forgotten Planet,Murray Leinster,1954,1954-00-00,Ace Books,1896,"Norfolk, Virginia, USA",0881846163,"**From the first page of the Ace Double:** ""Na...",...,0,0,False,False,USA,Central/North America,"[-0.025720153, 0.054792713, -0.021301318, 0.03...","[0.09217871, 0.109781586, 0.1129746, 0.1540808...",False,0
9,6101,The Star Beast,Robert A. Heinlein,1954,1954-08-23,Ace Books,1907,"Butler, Missouri, USA",0345275802,A talking alien pet has grown to the size of a...,...,0,0,False,False,USA,Central/North America,"[-0.03175946, 0.06456888, 0.05577806, -0.02333...","[0.07463385, 0.21729779, 0.19197693, 0.4434734...",False,0
11,1066774,The Wheels of Chance,H. G. Wells,1954,1954-00-00,J. M. Dent,1866,"Bromley, Kent, England, UK",0460019147,The comical Wheels of Chance was written in 18...,...,0,0,False,False,UK,Europe,"[-0.0009298305, 0.06744729, 0.03826583, 0.0196...","[0.13286704, 0.08107259, 0.11298126, -0.010794...",False,0
17,2266,The Caves of Steel,Isaac Asimov,1954,1954-00-00,HarperCollins (UK),1920,"Petrovichi, Smolensk Governorate, Russia",0586008357,"""A Del Rey book."" It was bad enough when Lije ...",...,0,0,False,False,Russia,Europe,"[-0.11344511, 0.007374352, 0.012599569, 0.0339...","[0.05855719, 0.06330989, 0.2975681, 0.26259395...",False,0


We start off by doing a simple test fit.

In [48]:
len(tags)

73

In [49]:
X_tt = books_tt.auto_tags.explode().values.astype(float).reshape(-1, 73)
locus_tt = books_tt.locus
hugo_tt = books_tt.hugo

X_val = books_val.auto_tags.explode().values.astype(float).reshape(-1, 73)
locus_val = books_val.locus
hugo_val = books_val.hugo

In [50]:
log_reg.fit(X_tt, locus_tt)
log_reg_balanced.fit(X_tt, locus_tt)

pred = log_reg.predict(X_val)
pred_balanced = log_reg.predict(X_val)

print('F1:', f1_score(locus_val, pred), f1_score(locus_val, pred_balanced))
print('Accuracy scores:', accuracy_score(locus_val, pred), accuracy_score(locus_val, pred_balanced)) 
print('Precision', precision_score(locus_val, pred), precision_score(locus_val, pred_balanced))

F1: 0.0 0.0
Accuracy scores: 0.9604415823367065 0.9604415823367065
Precision 0.0 0.0


/home/tiger/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/tiger/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [51]:
log_reg.fit(X_tt, hugo_tt)
log_reg_balanced.fit(X_tt, hugo_tt)

pred = log_reg.predict(X_val)
pred_balanced = log_reg.predict(X_val)

print('F1:', f1_score(hugo_val, pred), f1_score(hugo_val, pred_balanced))
print('Accuracy scores:', accuracy_score(hugo_val, pred), accuracy_score(hugo_val, pred_balanced)) 
print('Precision', precision_score(hugo_val, pred), precision_score(hugo_val, pred_balanced))

F1: 0.0 0.0
Accuracy scores: 0.995246856792395 0.995246856792395
Precision 0.0 0.0


/home/tiger/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/tiger/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


We see here that the auto-tagging system does not produce a very good model of winners. In fact, it seems that our logistic regression is just giving us the prediction of everything being false.

We can now try to do logistics regression with the previous author info along with the auto_tags.

In [29]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer

In [59]:
log_reg = Pipeline(steps = [
    ('features', 
        ColumnTransformer([
            ('pass', "passthrough", ['num_prev_awards'])
            ('auto_tags', FunctionTransformer(lambda x : x.explode().values.astype(float).reshape(-1, 73)), 'auto_tags'),
        ],
        remainder = "drop")
    ),
    ('logistic', LogisticRegression)
])

<>:4: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
<>:4: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
/tmp/ipykernel_13024/2666657065.py:4: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
  ('pass', "passthrough", ['num_prev_awards'])


TypeError: 'tuple' object is not callable

In [58]:
X_tt = books_tt[['auto_tags', 'num_prev_awards']]
y_tt = books_tt.target

log_reg.fit(X_tt,y_tt)

ValueError: setting an array element with a sequence.